# 06 - Train Support Vector Machine (SVM)

Train SVM classifier for toxicity prediction.

**Key Features:**
- RBF, linear, and polynomial kernels
- Feature standardization (important for SVM)
- C and gamma parameter optimization

In [ ]:

import os, sys, pickle
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

PARAM_GRID = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto'],
    'probability': [True],
    'class_weight': ['balanced', None]
}

TOXICITY_ENDPOINTS = ['NR-AhR', 'NR-AR', 'NR-AR-LBD', 'NR-Aromatase',
                      'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
                      'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']
MODELS_DIR = '../models/baseline_models'
print("✓ Setup complete")

In [ ]:

def train_svm(toxicity_name):
    """Train SVM for a single toxicity endpoint."""
    print(f"\nTraining SVM for {toxicity_name}...")
    
    cache_path = f'../Data/cache/{toxicity_name}/splits.pkl'
    if not os.path.exists(cache_path):
        return None
    
    with open(cache_path, 'rb') as f:
        data = pickle.load(f)
    
    # Standardize features (important for SVM!)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(data['train']['X'])
    X_val = scaler.transform(data['val']['X'])
    X_test = scaler.transform(data['test']['X'])
    
    X_train_val = np.vstack([X_train, X_val])
    y_train_val = np.concatenate([data['train']['y'], data['val']['y']])
    y_test = data['test']['y']
    
    # Grid search
    svm = SVC(random_state=42)
    grid_search = GridSearchCV(svm, PARAM_GRID, cv=5, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train_val, y_train_val)
    
    # Evaluate
    best_model = grid_search.best_estimator_
    y_proba = best_model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_proba)
    
    print(f"  Kernel={best_model.kernel}, C={best_model.C}, AUC: {test_auc:.4f}")
    
    # Save
    os.makedirs(f'{MODELS_DIR}/{toxicity_name}', exist_ok=True)
    with open(f'{MODELS_DIR}/{toxicity_name}/SVM_model.pkl', 'wb') as f:
        pickle.dump({'model': best_model, 'scaler': scaler}, f)
    
    return {'test_auc': test_auc}

result = train_svm('NR-AhR')